# STEP 2 — GPT 관련성 필터 (재현용)

STEP 1이 모은 논문 중 **mAb 안정성과 진짜 관련 있는 것만** GPT로 골라 `step2_filtered.csv` 로 저장합니다.

| | |
|---|---|
| **입력** | `step1_abstracts.csv` (STEP 1 출력) |
| **출력** | `step2_filtered.csv` (+ relevant, reason 컬럼) |
| **필요** | OpenAI API 키 |

> 🔑 Colab 왼쪽 🔑(보안 비밀)에 `OPENAI_API_KEY` 를 넣거나, 아래 셀 실행 시 입력하세요.
> 💰 `gpt-4o-mini` 기준 논문 수십 편 = 수십 원 수준. (원본 연구는 gpt-5-mini 사용)

In [ ]:
!pip install -q openai pandas
print('준비 완료')

In [ ]:
import os, json, time, pandas as pd
from getpass import getpass

# ── OpenAI 키 ──────────────────────────────────────────────
OPENAI_KEY = ''
try:
    from google.colab import userdata
    OPENAI_KEY = userdata.get('OPENAI_API_KEY') or ''
except Exception:
    OPENAI_KEY = os.environ.get('OPENAI_API_KEY', '')
if not OPENAI_KEY:
    OPENAI_KEY = getpass('OpenAI API key: ').strip()

from openai import OpenAI
client = OpenAI(api_key=OPENAI_KEY)
MODEL      = 'gpt-4o-mini'   # 저렴·범용 (원본: gpt-5-mini)
BATCH_SIZE = 10              # GPT 한 번에 판단할 논문 수
print('OpenAI 준비 완료, 모델:', MODEL)

In [ ]:
# 입력 불러오기 (STEP 1 출력)
import os
if not os.path.exists('step1_abstracts.csv'):
    raise FileNotFoundError('step1_abstracts.csv 가 없습니다. STEP 1을 먼저 실행하거나 파일을 업로드하세요.')
df_all = pd.read_csv('step1_abstracts.csv')
print(f'입력 논문: {len(df_all)}편')
df_all.head(2)

In [ ]:
SYSTEM_PROMPT = """You are an expert in monoclonal antibody (mAb) formulation and stability.

Your task: Judge whether each article contains MECHANISTIC information relevant to mAb stability.

RELEVANT (relevant=true) if the article covers ANY of:
- Physical stability of mAb: aggregation, viscosity, particle formation, turbidity, opalescence
- Chemical stability of mAb: oxidation, deamidation, isomerization, fragmentation, charge variants, glycosylation
- Biological stability: conformational stability, Tm, binding activity, potency, ADCC
- Effect of formulation conditions (excipients, pH, buffer, concentration) on mAb stability
- Effect of stress conditions (freeze-thaw, thermal, agitation, light, lyophilization) on stability
- Stability testing methods (SEC-HPLC, DLS, DSC, CE-SDS, etc.)
- Downstream consequences of instability: immunogenicity, ADA, PK changes, safety

NOT RELEVANT (relevant=false) if:
- mAb used only as a detection/research tool (e.g., ELISA reagent, flow cytometry)
- Clinical efficacy/trial with no stability content
- Stability of non-mAb molecules with mAb mentioned only peripherally

Respond ONLY with a JSON array, one object per article.
Format: [{"pmid": "...", "relevant": true/false, "reason": "one sentence"}]
No markdown, no text outside the JSON array."""

def filter_batch(articles):
    lines = [f'PMID: {a["pmid"]}\nTitle: {str(a["title"])[:200]}\nAbstract: {str(a["abstract"])[:500]}' for a in articles]
    user = '\n\n---\n\n'.join(lines)
    for attempt in range(3):
        try:
            r = client.chat.completions.create(model=MODEL,
                messages=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':user}],
                max_completion_tokens=2000)
            txt = r.choices[0].message.content.replace('```json','').replace('```','').strip()
            return json.loads(txt)
        except Exception as e:
            if attempt < 2: time.sleep(2)
            else:
                print('  batch 실패, 보수적으로 통과:', str(e)[:60])
                return [{'pmid':a['pmid'],'relevant':True,'reason':'filter_failed'} for a in articles]
print('필터 함수 정의 완료')

In [ ]:
from collections import OrderedDict
records = df_all.to_dict('records')
amap = {str(a['pmid']): a for a in records}
results = []
for i in range(0, len(records), BATCH_SIZE):
    batch = records[i:i+BATCH_SIZE]
    res = filter_batch(batch)
    results.extend(res)
    rel = sum(1 for r in res if r.get('relevant'))
    print(f'  {i+len(batch)}/{len(records)} 처리 · 이 배치 관련 {rel}/{len(batch)}')
    time.sleep(0.5)

# 결과를 원본에 합쳐 저장
rmap = {str(r.get('pmid')): r for r in results}
rows = []
for a in records:
    r = rmap.get(str(a['pmid']), {'relevant': True, 'reason': ''})
    rows.append({**a, 'relevant': 1 if r.get('relevant') else 0, 'relevance_reason': str(r.get('reason',''))[:300]})
out = pd.DataFrame(rows)
out.to_csv('step2_filtered.csv', index=False, encoding='utf-8-sig')
keep = int(out['relevant'].sum())
print(f'\n✅ 저장: step2_filtered.csv | 관련 {keep} / 전체 {len(out)} ({keep/len(out)*100:.0f}%)')
out[['pmid','relevant','relevance_reason']].head(6)

### ✅ STEP 2 완료
`step2_filtered.csv` 의 `relevant=1` 인 논문만 STEP 3에서 인과관계를 추출합니다.

**다음:** `step3_extract.ipynb`